# Day 20 — Errors, retries & budgets

Two things keep an agent from running forever or dying on a hiccup:

- a **step budget** (`max_steps`) so it can't loop endlessly, and
- a **resilient model call** so a transient timeout retries instead of crashing.

You'll wrap `chat()` with [`shared.reliability.resilient`](../../shared/reliability.py)
(timeout + retry) and fail **gracefully** when the budget or retries run out.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator
from shared.reliability import resilient

@resilient(attempts=3, time_budget=30.0)
def robust_chat(messages):
    return chat(messages, temperature=0)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def run(goal, tools, max_steps=6):
    messages = [
        {"role": "system", "content":
            'Reply ONLY JSON {"action":"calculator|finish","action_input":"..."}.'},
        {"role": "user", "content": goal},
    ]
    for _ in range(max_steps):
        # TODO 1: call robust_chat(messages) in try/except; on Exception return a graceful message
        # TODO 2: parse; if action == "finish" return action_input;
        #         else run tools[action](action_input) and append the Observation
        raise NotImplementedError
    return "stopped: step budget exhausted"

# print(run("What is 99 * 99?", {"calculator": calculator}))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator
from shared.reliability import resilient

@resilient(attempts=3, time_budget=30.0)
def robust_chat(messages):
    return chat(messages, temperature=0)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def run(goal, tools, max_steps=6):
    messages = [
        {"role": "system", "content":
            'Reply ONLY JSON {"action":"calculator|finish","action_input":"..."}.'},
        {"role": "user", "content": goal},
    ]
    for _ in range(max_steps):
        try:
            raw = robust_chat(messages)
        except Exception as exc:
            return f"giving up gracefully after retries: {exc}"
        obj = parse_json(raw)
        if obj.get("action") == "finish":
            return obj.get("action_input", "")
        tool = tools.get(obj.get("action"))
        obs = tool(obj.get("action_input", "")) if tool else f"unknown tool: {obj.get('action')}"
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "stopped: step budget exhausted"

print(run("What is 99 * 99?", {"calculator": calculator}))